In [19]:
import time
import cv2
import numba
import numpy as np
import pandas as pd
from typing import List
import matplotlib.pyplot as plt

# Entropy Function

In [20]:
def my_entropy(input_image):

    arr = input_image.flatten().astype(np.int64)

    if arr.min() != 1:
        arr = arr - arr.min() + 1

    p = np.zeros(arr.max(), dtype=np.float64)
    for v in arr:
        p[v - 1] += 1

    p = p / p.sum()
    p = p[p != 0]
    entropy = np.sum(-p * np.log2(p))

    return entropy

# PSNR Function

In [21]:
def next_power_of_two(x):
    return 1 << (x - 1).bit_length()

def peak_signal_noise_ratio(image1: np.ndarray, image2: np.ndarray):
    if image1.shape != image2.shape:
        return -1

    max_intensity = max(image1.max(), image2.max())

    next_pow2 = next_power_of_two(int(max_intensity + 1))
    max_value = next_pow2 - 1
    max_value_sq = max_value ** 2

    mse = np.mean((image1.astype(np.float64) - image2.astype(np.float64)) ** 2)

    if mse == 0:
        return np.inf

    return 10 * np.log10(max_value_sq / mse)

# Question1: Median Edge Predictor (MED)

In [22]:
def med_predictor(input_image):
    input_image = input_image.astype(np.int16)
    H, W = input_image.shape

    error_image = np.zeros((H, W), dtype=np.int16)

    padded_Image = np.pad(input_image, ((1, 0), (1, 0)), mode='constant', constant_values=0)

    for i in range(1, H + 1):
        for j in range(1, W + 1):

            a = padded_Image[i, j - 1]
            b = padded_Image[i - 1, j]
            c = padded_Image[i - 1, j - 1]

            if c >= max(a, b):
                x = min(a, b)
            elif c <= min(a, b):
                x = max(a, b)
            else:
                x = a + b - c

            error_image[i - 1, j - 1] = input_image[i - 1, j - 1] - x

    return error_image


def med_reconstructor(error_image):
    error_image = error_image.astype(np.int16)
    H, W = error_image.shape

    prediction = np.zeros((H + 1, W + 1), dtype=np.int16)
    reconstructed_image = np.zeros((H, W), dtype=np.uint8)

    for i in range(1, H + 1):
        for j in range(1, W + 1):
            a = prediction[i, j - 1]
            b = prediction[i - 1, j]
            c = prediction[i - 1, j - 1]

            if c >= max(a, b):
                x = min(a, b)
            elif c <= min(a, b):
                x = max(a, b)
            else:
                x = a + b - c

            prediction[i, j] = x + error_image[i - 1, j - 1]
            reconstructed_image[i - 1, j - 1] = np.uint8(prediction[i, j])

    return reconstructed_image

## MED Evaluation

In [23]:
test_path = ['Images/CLIC_2025_1.png', 'Images/CLIC_2025_2.png', 'Images/Kodak_01.png', 'Images/Kodak_23.png',
             'Images/Livingroom.tif', 'Images/Bridge.tif', 'Images/Baboon.tif', 'Images/Peppers.bmp',
             'Images/MRI_1.tif', 'Images/MRI_2.tif', 'Images/Retina.tif', 'Images/Cells.png']

evaluation_table_med = []
for image_name in test_path:
    name = image_name.split('/')[1].split('.')[0]
    image = cv2.imread(image_name, cv2.IMREAD_UNCHANGED)

    start_time = time.time()
    med_error = med_predictor(image)
    med_reconstruct = med_reconstructor(med_error)
    end_time = time.time()
    runtime = end_time - start_time

    evaluation_table_med.append({'Image Name': name,'Image Entropy': my_entropy(image),
                                 'Error Entropy(MED)': my_entropy(med_error), 'PSNR': peak_signal_noise_ratio(image, med_reconstruct),
                                 'Runtime': runtime})

evaluation_table_med = pd.DataFrame(evaluation_table_med)

# Average Row
error_entropy_mean_med = evaluation_table_med['Error Entropy(MED)'].mean()
runtime_mean_med = evaluation_table_med['Runtime'].mean()
evaluation_table_med.loc[len(evaluation_table_med)] = {"Image Name": "Average", 'Error Entropy(MED)': error_entropy_mean_med,
                                                       "PSNR": np.inf, "Runtime": runtime_mean_med}
evaluation_table_med

,Image Name,Image Entropy,Error Entropy(MED),PSNR,Runtime
0,CLIC_2025_1,7.573678,4.024232,inf,11.457403
1,CLIC_2025_2,7.615759,5.889772,inf,12.799804
2,Kodak_01,7.161006,5.511179,inf,1.738689
3,Kodak_23,7.251587,3.829204,inf,1.813003
4,Livingroom,7.295174,4.839180,inf,1.162918
5,Bridge,7.683018,5.668946,inf,1.308074
6,Baboon,7.292549,5.233969,inf,1.258011
7,Peppers,7.571478,4.843694,inf,1.171068
8,MRI_1,6.428016,3.765981,inf,3.988062
9,MRI_2,6.619796,3.612304,inf,4.213649


# Question2: Third Order Linear Predictor (Optimum Mode)

## Calculate Coefficients

In [78]:
def compute_third_order_coef(img):

    img = img.astype(np.float64)

    a = img[1:, :-1].flatten()
    b = img[:-1, 1:].flatten()
    c = img[:-1, :-1].flatten()
    x = img[1:, 1:].flatten()

    A = np.sum(a*a)
    B = np.sum(b*b)
    C = np.sum(c*c)
    D = np.sum(a*b)
    E = np.sum(a*c)
    F = np.sum(b*c)

    Xa = np.sum(a*x)
    Xb = np.sum(b*x)
    Xc = np.sum(c*x)

    M = np.array([
        [A, D, E, 1],
        [D, B, F, 1],
        [E, F, C, 1],
        [1, 1, 1, 0]
    ], dtype=np.float64)

    rhs = np.array([Xa, Xb, Xc, 1], dtype=np.float64)
    sol = np.linalg.solve(M, rhs)

    alpha, beta = sol[0], sol[1]
    return alpha, beta

## Third Order Optimum Predictor

In [82]:
def third_optimum_predictor(input_image):
    input_image = input_image.astype(dtype=np.float64)
    H, W = input_image.shape

    error_image = np.zeros((H, W), dtype=np.float64)
    alpha, beta = compute_third_order_coef(input_image)
    gamma = 1.0 - (alpha + beta)

    padded_Image = np.pad(input_image, ((1, 0), (1, 0)), mode='constant', constant_values=0)

    for i in range(1, H + 1):
        for j in range(1, W + 1):

            a = padded_Image[i, j - 1]
            b = padded_Image[i - 1, j]
            c = padded_Image[i - 1, j - 1]
            x = (alpha*a) + (beta*b) + (gamma*c)

            error_image[i - 1, j - 1] = input_image[i - 1, j - 1] - x

    return error_image, alpha, beta


def third_optimum_reconstructor(error_image, alpha, beta):
    error_image = error_image.astype(np.float64)
    H, W = error_image.shape
    gamma = 1.0 - (alpha + beta)

    prediction = np.zeros((H + 1, W + 1), dtype=np.float64)
    reconstructed_image = np.zeros((H, W), dtype=np.float64)

    for i in range(1, H + 1):
        for j in range(1, W + 1):
            a = prediction[i, j - 1]
            b = prediction[i - 1, j]
            c = prediction[i - 1, j - 1]

            x = (alpha*a) + (beta*b) + (gamma*c)

            prediction[i, j] = x + error_image[i - 1, j - 1]
            reconstructed_image[i - 1, j - 1] = prediction[i, j]

    return reconstructed_image

## Evaluation of third Order Optimum Predictor

In [83]:
evaluation_table_opt3 = []

for image_name in test_path:
    name = image_name.split('/')[1].split('.')[0]
    image = cv2.imread(image_name, cv2.IMREAD_UNCHANGED)

    start_time = time.time()
    three_opt_error, alpha, beta = third_optimum_predictor(image)
    three_opt_reconstruct = third_optimum_reconstructor(three_opt_error, alpha, beta)
    end_time = time.time()
    runtime = end_time - start_time

    evaluation_table_opt3.append({'Image Name': name, 'Error Entropy(OPT-3)': my_entropy(three_opt_error),
                                  'PSNR': peak_signal_noise_ratio(image, three_opt_reconstruct),
                                  'Runtime': runtime})

evaluation_table_opt3 = pd.DataFrame(evaluation_table_opt3)

# Average Row
error_entropy_mean_opt3 = evaluation_table_opt3['Error Entropy(OPT-3)'].mean()
runtime_mean_opt3 = evaluation_table_opt3['Runtime'].mean()
evaluation_table_opt3.loc[len(evaluation_table_opt3)] = {"Image Name": "Average", 'Error Entropy(OPT-3)': error_entropy_mean_opt3,
                                                         "PSNR": np.inf,
                                                         "Runtime": runtime_mean_opt3}

evaluation_table_opt3

,Image Name,Error Entropy(OPT-3),PSNR,Runtime
0,CLIC_2025_1,3.572794,381.918793,4.246247
1,CLIC_2025_2,5.739044,inf,4.532629
2,Kodak_01,5.511926,inf,0.642854
3,Kodak_23,3.543218,inf,0.642195
4,Livingroom,4.830203,310.046808,0.421427
5,Bridge,5.599623,378.240748,0.418192
6,Baboon,5.015353,inf,0.421956
7,Peppers,4.712849,376.289565,0.422177
8,MRI_1,3.204443,285.439082,0.417010
9,MRI_2,2.975721,307.270460,0.420218


# Question3: Designing Predictor

## Method 1: Partitioning & MED
In this method, we combine the MED (Median Edge Detector) predictor with a partitioning technique for image prediction and reconstruction. The core idea is to predict each pixel’s value based on its neighbors and then encode the prediction error, which often results in better compression efficiency.

First, the image is divided into multiple partitions. In the code, we experimented with splitting the image into 4 and 2 partitions, though the current implementation uses 16 as an example. Each partition is processed independently, which allows the predictor to adapt to local image characteristics and improves overall prediction accuracy.

In [25]:
def med_predict(a, b, c):
    if c >= max(a, b):
        return min(a, b)
    elif c <= min(a, b):
        return max(a, b)
    else:
        return a + b - c


def my_predictor(input_image):
    img = input_image.astype(np.float64)

    partitions = np.array_split(img, 16, axis=0)
    error_partitions = []

    for part in partitions:
        h, w = part.shape
        err = np.zeros((h, w), dtype=np.float64)

        padded = np.pad(part, ((1, 0), (1, 0)), mode='constant')

        for i in range(1, h + 1):
            for j in range(1, w + 1):
                a = padded[i, j - 1]
                b = padded[i - 1, j]
                c = padded[i - 1, j - 1]

                pred = med_predict(a, b, c)
                err[i - 1, j - 1] = part[i - 1, j - 1] - pred

        error_partitions.append(err)

    error_image = np.vstack(error_partitions)
    return error_image

def my_reconstructor(error_image):
    err = error_image.astype(np.float64)
    err_parts = np.array_split(err, 16, axis=0)

    reconstructed_parts = []

    for err_part in err_parts:
        h, w = err_part.shape

        pred_img = np.zeros((h + 1, w + 1), dtype=np.float64)
        rec = np.zeros((h, w), dtype=np.float64)

        for i in range(1, h + 1):
            for j in range(1, w + 1):
                a = pred_img[i, j - 1]
                b = pred_img[i - 1, j]
                c = pred_img[i - 1, j - 1]

                pred = med_predict(a, b, c)

                val = pred + err_part[i - 1, j - 1]
                pred_img[i, j] = val
                rec[i - 1, j - 1] = val

        reconstructed_parts.append(rec)

    reconstructed_image = np.vstack(reconstructed_parts)
    return reconstructed_image

## Evaluation of Method 1

In [26]:
evaluation_table_one = []

for image_name in test_path:
    name = image_name.split('/')[1].split('.')[0]
    image = cv2.imread(image_name, cv2.IMREAD_UNCHANGED)

    start_time = time.time()
    my_predictor_error = my_predictor(image)
    my_predictor_reconstruct = my_reconstructor(my_predictor_error)
    end_time = time.time()
    runtime = end_time - start_time

    evaluation_table_one.append({'Image Name': name, 'Error Entropy(MED-with-Partitioning)': my_entropy(my_predictor_error),
                                 'PSNR': peak_signal_noise_ratio(image, my_predictor_reconstruct),
                                 'Runtime': runtime})

evaluation_table_one = pd.DataFrame(evaluation_table_one)

# Average Row
error_entropy_mean_one = evaluation_table_one['Error Entropy(MED-with-Partitioning)'].mean()
runtime_mean_one = evaluation_table_one['Runtime'].mean()
evaluation_table_one.loc[len(evaluation_table_one)] = {"Image Name": "Average", 'Error Entropy(MED-with-Partitioning)': error_entropy_mean_one,
                                                       "PSNR": np.inf, "Runtime": runtime_mean_one}

evaluation_table_one

,Image Name,Error Entropy(MED-with-Partitioning),PSNR,Runtime
0,CLIC_2025_1,4.026119,inf,12.816276
1,CLIC_2025_2,5.892087,inf,31.066098
2,Kodak_01,5.523678,inf,4.043013
3,Kodak_23,3.840914,inf,4.136998
4,Livingroom,4.861838,inf,2.705801
5,Bridge,5.681421,inf,2.666946
6,Baboon,5.254854,inf,2.782887
7,Peppers,4.850672,inf,2.705255
8,MRI_1,3.794911,inf,2.968547
9,MRI_2,3.642836,inf,0.944021


## Method2: Second Order Optimum LS Predictor with Partitioning
In this method, we combine a linear predictor with <b>adaptive coefficients and a partitioning technique</b>. The image is divided into <b>several horizontal partitions, and for each partition, a coefficient `ρ`</b> is computed to best capture the local correlation between neighboring pixels. Each pixel is then predicted as `pred = aρ + (1-ρ)*b = b + ρ*(b - a)`, where a and b are the left and top neighbors, respectively, and the prediction error is stored. During reconstruction, the same coefficient is used for each partition, and the original pixels are recovered by adding the prediction errors back to the predicted values. By adapting the predictor to each partition, this method reduces prediction error and improves reconstruction accuracy compared to using a fixed predictor.

In [29]:
def compute_coefficients(part):

    part = part.astype(np.float64)

    a = part[1:, :-1].ravel()
    b = part[:-1, 1:].ravel()
    x = part[1:, 1:].ravel()

    diff = (b - a)
    denom = np.sum(diff * diff)
    if denom == 0:
        return 0.0
    numer = np.sum((x - b) * diff)
    rho = numer / denom
    return float(rho)


def my_predictor_second(input_image, n_partitions=4):

    img = input_image.astype(np.float64)
    parts = np.array_split(img, n_partitions, axis=0)

    error_parts = []
    coef_list = []

    for part in parts:
        h, w = part.shape
        coef = compute_coefficients(part)
        coef_list.append(coef)

        padded = np.pad(part, ((1, 0), (1, 0)), mode='constant', constant_values=0.0)
        err = np.zeros((h, w), dtype=np.float64)

        for i in range(1, h + 1):
            for j in range(1, w + 1):
                a = padded[i, j - 1]
                b = padded[i - 1, j]
                pred = b + coef*(b - a)
                err[i - 1, j - 1] = part[i - 1, j - 1] - pred

        error_parts.append(err)

    error_image = np.vstack(error_parts)
    return error_image, coef_list


def my_reconstructor_second(error_image, coef_list):
    err = error_image
    err_parts = np.array_split(err, len(coef_list), axis=0)

    recon_parts = []

    for err_part, rho in zip(err_parts, coef_list):
        h, w = err_part.shape
        pred_buf = np.zeros((h + 1, w + 1), dtype=np.float64)
        rec = np.zeros((h, w), dtype=np.float64)

        for i in range(1, h + 1):
            for j in range(1, w + 1):
                a = pred_buf[i, j - 1]
                b = pred_buf[i - 1, j]
                pred = b + rho*(b - a)
                val = pred + err_part[i - 1, j - 1]
                pred_buf[i, j] = val
                rec[i - 1, j - 1] = val

        recon_parts.append(rec)

    reconstructed = np.vstack(recon_parts)
    return reconstructed

## Evaluation of Method 2

In [30]:
def bpp_for_coefficients(image_shape, n_partitions, bits_per_coef=64):
    H, W = image_shape
    total_bits = n_partitions * bits_per_coef
    bpp = total_bits / (H * W)
    return bpp

In [31]:
evaluation_table_two = []
for image_name in test_path:
    name = image_name.split('/')[1].split('.')[0]
    image = cv2.imread(image_name, cv2.IMREAD_UNCHANGED)

    start_time = time.time()
    my_predictor_error_second, coef = my_predictor_second(image, n_partitions=2)
    reconstruct_second = my_reconstructor_second(my_predictor_error_second, coef)
    end_time = time.time()
    runtime = end_time - start_time

    evaluation_table_two.append({'Image Name': name, 'Error Entropy(Second Order-Partitioning)': my_entropy(my_predictor_error_second),
                                 'PSNR': peak_signal_noise_ratio(image, reconstruct_second),
                                 'BPP': bpp_for_coefficients(image.shape, n_partitions=2, bits_per_coef=64),
                                 'Runtime': runtime})

evaluation_table_two = pd.DataFrame(evaluation_table_two)


# Average Row
error_entropy_mean_two = evaluation_table_two['Error Entropy(Second Order-Partitioning)'].mean()
runtime_mean_two = evaluation_table_two['Runtime'].mean()
bpp_mean_two = evaluation_table_two['BPP'].mean()
evaluation_table_two.loc[len(evaluation_table_two)] = {"Image Name": "Average", 'Error Entropy(Second Order-Partitioning)': error_entropy_mean_two,
                                                       "PSNR": np.inf,
                                                       "BPP": bpp_mean_two,
                                                       "Runtime": runtime_mean_two}

evaluation_table_two

,Image Name,Error Entropy(Second Order-Partitioning),PSNR,BPP,Runtime
0,CLIC_2025_1,3.689904,inf,0.000051,5.992718
1,CLIC_2025_2,5.821493,inf,0.000046,5.786705
2,Kodak_01,5.665907,inf,0.000326,0.861200
3,Kodak_23,3.627043,inf,0.000326,0.908237
4,Livingroom,5.009845,inf,0.000488,0.525642
5,Bridge,5.707580,inf,0.000488,0.623387
6,Baboon,5.378950,inf,0.000488,0.618652
7,Peppers,4.649051,inf,0.000488,0.677542
8,MRI_1,4.005337,inf,0.000488,0.655094
9,MRI_2,3.683289,inf,0.000488,0.603151


## Method 3: Med with Context Modeling
In this final method, we enhance the classical MED predictor by integrating a <b>four-class context</b> modeling system that captures local structural variations in the image. For every pixel, we look at its neighbors `a (left)`, `b (top)`, and `c (top-left)`, and <b>classify the pixel into one of four contexts</b> based on the relative differences `|a−c|` and `|b−c|`:
- smooth regions (context 0)
- vertical edges (context 1)
- horizontal edges (context 2)
- textured or complex regions (context 3).

Using these contexts, the MED predictor is applied in a fully vectorized manner to generate a predicted value for each pixel. Prediction residuals (the difference between the original pixel and the prediction) are then separated into four lists, one for each context, using <b>a fast Numba-optimized</b> routine. Because each context groups together pixels with similar structural behavior, the residuals within each context exhibit a narrower and more regular statistical distribution, leading to lower entropy and better compression efficiency. During reconstruction, the residuals are fed back into a MED-based decoder that processes pixels in raster scan order, selecting the appropriate residual from its context-specific list and adding it to the corresponding MED prediction. This ensures a correct and lossless reconstruction while preserving the benefits gained from context-based entropy reduction.

### BPP calculation fot method 3

In [39]:
def bpp_for_context_modeling(residuals_by_context: List[np.ndarray], contexts: np.ndarray, image_shape: tuple) -> float:
    H, W = image_shape
    total_pixels = H * W

    # Calculate bits for residuals (using entropy coding estimate)
    residual_bits = 0
    for i, residuals in enumerate(residuals_by_context):
        if len(residuals) > 0:
            hist, bin_edges = np.histogram(residuals, bins=256, density=True)
            hist = hist[hist > 0]
            entropy = -np.sum(hist * np.log2(hist))

            residual_bits += len(residuals) * entropy

    # Calculate bits for context map
    contexts_flat = contexts.ravel()
    hist_context, _ = np.histogram(contexts_flat, bins=4, range=(0, 3), density=True)
    hist_context = hist_context[hist_context > 0]
    context_entropy = -np.sum(hist_context * np.log2(hist_context))
    context_bits = total_pixels * context_entropy

    # Total bits and BPP
    total_bits = residual_bits + context_bits
    bpp = total_bits / total_pixels

    return bpp

### Conditional Entropy Calculation

In [40]:
def entropy_from_values(values):
    if len(values) == 0:
        return 0.0

    values = np.array(values, dtype=np.int64)
    unique, counts = np.unique(values, return_counts=True)
    probs = counts / counts.sum()

    return -np.sum(probs * np.log2(probs))


def entropy_for_context_model(residuals_by_context):

    entropy_per_ctx = []
    total_count = sum(len(resid_list) for resid_list in residuals_by_context)

    weighted_sum = 0.0
    for ctx_id, resid_list in enumerate(residuals_by_context):
        H_ctx = entropy_from_values(resid_list)
        entropy_per_ctx.append(H_ctx)

        p = len(resid_list) / total_count if total_count > 0 else 0
        weighted_sum += p * H_ctx

    return entropy_per_ctx, weighted_sum

In [41]:
def classify_context_vectorized(a: np.ndarray, b: np.ndarray, c: np.ndarray, threshold: int = 4):

    d1 = np.abs(a - c)
    d2 = np.abs(b - c)

    # Smooth Region as Default
    contexts = np.zeros_like(a, dtype=np.int8)

    # Vertical edge
    mask_vertical = (d1 >= threshold) & (d2 < threshold)
    contexts[mask_vertical] = 1

    # Horizontal edge
    mask_horizontal = (d1 < threshold) & (d2 >= threshold)
    contexts[mask_horizontal] = 2

    # Textured Region
    mask_textured = (d1 >= threshold) & (d2 >= threshold)
    contexts[mask_textured] = 3

    return contexts

def predict_med_vectorized(a: np.ndarray, b: np.ndarray, c: np.ndarray):
    pred = np.zeros_like(a)

    mask1 = (c >= np.maximum(a, b))
    pred[mask1] = np.minimum(a[mask1], b[mask1])

    mask2 = (c <= np.minimum(a, b))
    pred[mask2] = np.maximum(a[mask2], b[mask2])

    mask3 = ~(mask1 | mask2)
    pred[mask3] = a[mask3] + b[mask3] - c[mask3]

    return pred

@numba.njit
def extract_residuals_by_context_numba(residuals_flat: np.ndarray, contexts_flat: np.ndarray,
                                       total_pixels: int) :
    residuals_0 = np.zeros(total_pixels, dtype=residuals_flat.dtype)
    residuals_1 = np.zeros(total_pixels, dtype=residuals_flat.dtype)
    residuals_2 = np.zeros(total_pixels, dtype=residuals_flat.dtype)
    residuals_3 = np.zeros(total_pixels, dtype=residuals_flat.dtype)

    count_0 = 0
    count_1 = 0
    count_2 = 0
    count_3 = 0

    for i in range(len(residuals_flat)):
        ctx = contexts_flat[i]
        resid = residuals_flat[i]

        if ctx == 0:
            residuals_0[count_0] = resid
            count_0 += 1
        elif ctx == 1:
            residuals_1[count_1] = resid
            count_1 += 1
        elif ctx == 2:
            residuals_2[count_2] = resid
            count_2 += 1
        else:
            residuals_3[count_3] = resid
            count_3 += 1

    return (residuals_0[:count_0], residuals_1[:count_1],
            residuals_2[:count_2], residuals_3[:count_3])

def my_predictor_third(image: np.ndarray):
    image = image.astype(np.float64)
    H, W = image.shape

    padded = np.zeros((H + 1, W + 1), dtype=np.float64)
    padded[1:, 1:] = image

    a = padded[1:, :-1]
    b = padded[:-1, 1:]
    c = padded[:-1, :-1]

    contexts = classify_context_vectorized(a, b, c)
    pred = predict_med_vectorized(a, b, c)

    residuals = image - pred
    residuals_0, residuals_1, residuals_2, residuals_3 = extract_residuals_by_context_numba(
        residuals.ravel(), contexts.ravel(), H * W
    )

    return [residuals_0, residuals_1, residuals_2, residuals_3], contexts

@numba.njit
def med_with_context_decoder_optimized_numba(residuals_by_context: List[np.ndarray], contexts: np.ndarray) -> np.ndarray:

    H, W = contexts.shape
    image = np.zeros((H + 1, W + 1), dtype=np.float64)

    context_counts = np.zeros(4, dtype=np.int32)
    for ctx_val in contexts.ravel():
        context_counts[ctx_val] += 1

    for ctx in range(4):
        if len(residuals_by_context[ctx]) != context_counts[ctx]:
            raise ValueError(f"Context counts mismatch for context {ctx}")

    resid_arrays = [residuals_by_context[i] for i in range(4)]
    resid_indices = np.zeros(4, dtype=np.int32)

    for i in range(1, H + 1):
        for j in range(1, W + 1):
            a = image[i, j - 1]
            b = image[i - 1, j]
            c = image[i - 1, j - 1]

            ctx = contexts[i - 1, j - 1]
            resid = resid_arrays[ctx][resid_indices[ctx]]
            resid_indices[ctx] += 1

            if c >= max(a, b):
                pred = min(a, b)
            elif c <= min(a, b):
                pred = max(a, b)
            else:
                pred = a + b - c

            image[i, j] = pred + resid

    return image[1:, 1:]

def my_reconstruct_third(residuals_by_context: List[np.ndarray], contexts: np.ndarray):
    return med_with_context_decoder_optimized_numba(residuals_by_context, contexts)

## Evaluation of Method 3

In [42]:
evaluation_table_three = []
for image_name in test_path:
    name = image_name.split('/')[1].split('.')[0]
    image = cv2.imread(image_name, cv2.IMREAD_UNCHANGED)

    start_time = time.time()
    residuals, contexts = my_predictor_third(image)
    reconstructed = my_reconstruct_third(residuals, contexts)
    end_time = time.time()
    runtime = end_time - start_time

    entropy_per_ctx, overall_entropy = entropy_for_context_model(residuals)

    evaluation_table_three.append({'Image Name': name, 'Error Entropy(Med with Context)': overall_entropy,
                                   'PSNR': peak_signal_noise_ratio(image, reconstructed),
                                   'BPP': bpp_for_context_modeling(residuals_by_context=residuals, contexts=contexts, image_shape=image.shape),
                                   'Runtime': runtime})

evaluation_table_three = pd.DataFrame(evaluation_table_three)


# Average Row
error_entropy_mean_three = evaluation_table_three['Error Entropy(Med with Context)'].mean()
runtime_mean_three = evaluation_table_three['Runtime'].mean()
bpp_mean_three = evaluation_table_three['BPP'].mean()
evaluation_table_three.loc[len(evaluation_table_three)] = {"Image Name": "Average", 'Error Entropy(Med with Context)': error_entropy_mean_three,
                                                           "PSNR": np.inf,
                                                           "BPP": bpp_mean_three,
                                                           "Runtime": runtime_mean_three}

evaluation_table_three

,Image Name,Error Entropy(Med with Context),PSNR,BPP,Runtime
0,CLIC_2025_1,3.607005,inf,5.874918,4.981867
1,CLIC_2025_2,5.706341,inf,6.836424,0.520654
2,Kodak_01,5.350421,inf,6.503722,0.075202
3,Kodak_23,3.672004,inf,5.670970,0.058126
4,Livingroom,4.784764,inf,7.603010,0.051583
5,Bridge,5.596805,inf,7.743349,0.037998
6,Baboon,5.136580,inf,9.285636,0.039489
7,Peppers,4.766728,inf,8.438841,0.041069
8,MRI_1,3.419051,inf,6.563298,0.027058
9,MRI_2,3.320447,inf,5.714222,0.028986


In [85]:
print(evaluation_table_med['Error Entropy(MED)'].mean())
print(evaluation_table_opt3['Error Entropy(OPT-3)'].mean())
print(evaluation_table_one['Error Entropy(MED-with-Partitioning)'].mean())
print(evaluation_table_two['Error Entropy(Second Order-Partitioning)'].mean())
print(evaluation_table_three['Error Entropy(Med with Context)'].mean())

4.493368715834052
4.217094265647606
4.507836740963415
4.453061349360311
4.29394541391275
